In [6]:
import os
import json
import freesound

In [7]:
FREESOUND_STORE_METADATA_FIELDS = ['id', 'description', 'name', 'username', 'previews', 'license', 'tags']

In [8]:
FREESOUND_API_KEY = os.getenv('FREESOUND_API_KEY')
FREESOUND_API_KEY = FREESOUND_API_KEY.strip('"')

client = freesound.FreesoundClient()
client.set_token(FREESOUND_API_KEY)

In [9]:
ANNOTATIONS_PATH = "../data/annotations5.json"
annotations = None
with open(ANNOTATIONS_PATH, "r") as f:
    annotations = json.load(f)

In [17]:
AUDIO_DIR = "../data/audio/"
META_DIR = "../data/meta/"


In [12]:
import os
import json
import glob
import time
import math
import pandas as pd
import freesound

# 1. API Client Setup
FREESOUND_API_KEY = os.getenv('FREESOUND_API_KEY', '').strip('"')
client = freesound.FreesoundClient()
client.set_token(FREESOUND_API_KEY)

ANNOTATIONS_PATH = "../data/annotations5.json"
META_DIR = "../data/metadata"
OUTPUT_PATH = "../data/complete_annotations.json"

# 2. Map Sound IDs to Search Queries via Metadata CSVs
sound_to_query = {}
meta_files = glob.glob(os.path.join(META_DIR, "*.csv"))

for meta_path in meta_files:
    query_name = os.path.basename(meta_path).replace(".csv", "").replace("_", " ")
    json_query_path = meta_path.rsplit(".", 1)[0] + ".json"
    
    if os.path.exists(json_query_path):
        with open(json_query_path, 'r') as qf:
            q_data = json.load(qf)
            query_name = q_data.get('query', query_name)

    df = pd.read_csv(meta_path)
    if 'sound_id' in df.columns:
        for sid in df['sound_id']:
            sound_to_query[str(sid)] = query_name

# 3. Load Base Annotations
with open(ANNOTATIONS_PATH, "r") as f:
    annotations = json.load(f)

all_ids = list(annotations.keys())
total_sounds = len(all_ids)
BATCH_SIZE = 100  # Freesound max page size
metadata_cache = {}

print(f"Batch-fetching metadata for {total_sounds} records across {math.ceil(total_sounds / BATCH_SIZE)} API requests...")

# 4. Fetch Metadata in Batches of 100
for i in range(0, total_sounds, BATCH_SIZE):
    batch_ids = all_ids[i:i + BATCH_SIZE]
    
    # Construct an ID filter query: "id:(123 OR 456 OR 789)"
    id_filter = "id:(" + " OR ".join(batch_ids) + ")"
    
    try:
        # Standard search call in freesound-python wrapper
        results = client.search(
            filter=id_filter,
            fields="id,name,username,license",
            page_size=BATCH_SIZE
        )
        
        # Iterate over results pager
        retrieved_count = 0
        for sound in results:
            metadata_cache[str(sound.id)] = {
                "title": getattr(sound, 'name', 'Unknown'),
                "author": getattr(sound, 'username', 'Unknown'),
                "license": getattr(sound, 'license', 'Unknown')
            }
            retrieved_count += 1
            
        print(f"Batch {i // BATCH_SIZE + 1}/{(total_sounds + BATCH_SIZE - 1) // BATCH_SIZE} completed ({retrieved_count} sounds retrieved)")
        
    except Exception as e:
        print(f"Error fetching batch {i // BATCH_SIZE + 1}: {e}")
    
    # Respect rate limits (1 second pause between batches)
    time.sleep(1.0)

# 5. Build Final Enriched Dataset
enriched_dataset = {}

for sound_id, annotation_data in annotations.items():
    meta = metadata_cache.get(str(sound_id), {
        "title": "Unknown",
        "author": "Unknown",
        "license": "Unknown"
    })
    
    enriched_dataset[str(sound_id)] = {
        "title": meta["title"],
        "author": meta["author"],
        "freesound_url": f"https://freesound.org/s/{sound_id}/",
        "license": meta["license"],
        "search_query": sound_to_query.get(str(sound_id), "unknown"),
        "annotations": {
            "repetitive_onset_indices": annotation_data.get("repetitive_onset_indices", []),
            "repetitive_onset_times": annotation_data.get("repetitive_onset_times", [])
        }
    }

# 6. Save JSON
with open(OUTPUT_PATH, "w") as f:
    json.dump(enriched_dataset, f, indent=2)

print(f"\nDone! Enriched dataset exported to {OUTPUT_PATH}")

Batch-fetching metadata for 1028 records across 11 API requests...
Batch 1/11 completed (100 sounds retrieved)
Batch 2/11 completed (100 sounds retrieved)
Batch 3/11 completed (100 sounds retrieved)
Batch 4/11 completed (100 sounds retrieved)
Batch 5/11 completed (100 sounds retrieved)
Batch 6/11 completed (100 sounds retrieved)
Batch 7/11 completed (100 sounds retrieved)
Batch 8/11 completed (100 sounds retrieved)
Batch 9/11 completed (100 sounds retrieved)
Batch 10/11 completed (100 sounds retrieved)
Batch 11/11 completed (28 sounds retrieved)

Done! Enriched dataset exported to ../data/enriched_annotations.json


In [14]:
annotations = None
with open(OUTPUT_PATH, "r") as f:
    annotations = json.load(f)

In [15]:
annotations

{'655124': {'title': 'Dog Barking 3 times.wav',
  'author': '221339',
  'freesound_url': 'https://freesound.org/s/655124/',
  'license': 'https://creativecommons.org/licenses/by/4.0/',
  'search_query': 'unknown',
  'annotations': {'repetitive_onset_indices': [0, 1, 2],
   'repetitive_onset_times': [0.5340589284896851,
    2.2639455795288086,
    3.8196825981140137]}},
 '722978': {'title': 'Wood Knocking 3 Times',
  'author': 'SamsCollegeAccount',
  'freesound_url': 'https://freesound.org/s/722978/',
  'license': 'http://creativecommons.org/publicdomain/zero/1.0/',
  'search_query': 'unknown',
  'annotations': {'repetitive_onset_indices': [0, 1, 2],
   'repetitive_onset_times': [0.18575963377952576,
    0.847528338432312,
    1.5092970132827759]}},
 '765153': {'title': 'Gun Shot 3 Times',
  'author': 'MieckevanHoek',
  'freesound_url': 'https://freesound.org/s/765153/',
  'license': 'http://creativecommons.org/publicdomain/zero/1.0/',
  'search_query': 'unknown',
  'annotations': {'rep

Search query is incorrectly assigned. Reassign from the combined dataset

In [28]:
df = None
print("Found metadata sources:")
for csv_path in glob.glob(os.path.join(META_DIR, "dataset_*.csv")):
    print(csv_path)

    # figure out the used query based on the query metadata file
    json_path = csv_path[:csv_path.rfind(".")] + ".json"
    with open(json_path, "r") as f:
        query_data = json.load(f)
        
    raw_query = query_data.get("query", "")
    
    # 1. Split at the first " -" to drop all exclusions, and strip any trailing whitespace
    clean_query = raw_query.split(" -")[0].strip()
    
    # 2. Swap double quotes for single quotes to prevent ugly CSV triple-quoting
    # '"3 times"' becomes "'3 times'", 'dog barking' remains 'dog barking'
    clean_query = clean_query.replace('"', "'")
    
    if df is None:
        df = pd.read_csv(csv_path)
        df["search_query"] = clean_query
    else:
        local_df = pd.read_csv(csv_path)
        local_df["search_query"] = clean_query
        df = pd.concat([df, local_df], ignore_index=True)
        
df = df.drop_duplicates(subset=['sound_id'], keep='first').reset_index(drop=True)

Found metadata sources:
../data/meta/dataset_2026-06-15_14-36-36.csv
../data/meta/dataset_2026-05-20_14-27-00.csv
../data/meta/dataset_2026-06-02_15-32-39.csv
../data/meta/dataset_2026-06-02_16-15-49.csv
../data/meta/dataset_2026-06-12_18-50-48.csv
../data/meta/dataset_2026-06-15_15-18-42.csv
../data/meta/dataset_2026-06-02_18-16-48.csv
../data/meta/dataset_2026-06-13_16-52-26.csv
../data/meta/dataset_2026-06-02_16-24-32.csv
../data/meta/dataset_2026-06-14_09-39-52.csv
../data/meta/dataset_2026-06-13_17-47-59.csv
../data/meta/dataset_2026-06-02_17-59-57.csv
../data/meta/dataset_2026-06-02_17-13-17.csv
../data/meta/dataset_2026-05-23_14-14-52.csv
../data/meta/dataset_2026-06-12_18-26-39.csv
../data/meta/dataset_2026-05-23_14-36-27.csv
../data/meta/dataset_2026-06-02_17-08-15.csv
../data/meta/dataset_2026-06-02_16-21-22.csv
../data/meta/dataset_2026-05-23_14-36-09.csv
../data/meta/dataset_2026-06-12_18-54-51.csv
../data/meta/dataset_2026-06-02_15-40-15.csv
../data/meta/dataset_2026-06-12

In [27]:
for i in range(len(df)):
    row = df.loc[i]
    print(f'{i}. Replacing {annotations[str(row["sound_id"])]["search_query"]} with {row["search_query"]}')
    annotations[str(row["sound_id"])]["search_query"] = row["search_query"]

0. Replacing unknown with propeller
1. Replacing unknown with propeller
2. Replacing unknown with propeller
3. Replacing unknown with propeller
4. Replacing unknown with propeller
5. Replacing unknown with propeller
6. Replacing unknown with propeller
7. Replacing unknown with propeller
8. Replacing unknown with propeller
9. Replacing unknown with propeller
10. Replacing unknown with propeller
11. Replacing unknown with propeller
12. Replacing unknown with propeller
13. Replacing unknown with propeller
14. Replacing unknown with propeller
15. Replacing unknown with propeller
16. Replacing unknown with propeller
17. Replacing unknown with propeller
18. Replacing unknown with propeller
19. Replacing unknown with propeller
20. Replacing unknown with propeller
21. Replacing unknown with propeller
22. Replacing unknown with propeller
23. Replacing unknown with propeller
24. Replacing unknown with propeller
25. Replacing unknown with propeller
26. Replacing unknown with '3 times'
27. Replaci

In [29]:
annotations

{'655124': {'title': 'Dog Barking 3 times.wav',
  'author': '221339',
  'freesound_url': 'https://freesound.org/s/655124/',
  'license': 'https://creativecommons.org/licenses/by/4.0/',
  'search_query': "'3 times'",
  'annotations': {'repetitive_onset_indices': [0, 1, 2],
   'repetitive_onset_times': [0.5340589284896851,
    2.2639455795288086,
    3.8196825981140137]}},
 '722978': {'title': 'Wood Knocking 3 Times',
  'author': 'SamsCollegeAccount',
  'freesound_url': 'https://freesound.org/s/722978/',
  'license': 'http://creativecommons.org/publicdomain/zero/1.0/',
  'search_query': "'3 times'",
  'annotations': {'repetitive_onset_indices': [0, 1, 2],
   'repetitive_onset_times': [0.18575963377952576,
    0.847528338432312,
    1.5092970132827759]}},
 '765153': {'title': 'Gun Shot 3 Times',
  'author': 'MieckevanHoek',
  'freesound_url': 'https://freesound.org/s/765153/',
  'license': 'http://creativecommons.org/publicdomain/zero/1.0/',
  'search_query': "'3 times'",
  'annotations':

In [31]:
with open(OUTPUT_PATH, "w") as f:
    json.dump(annotations, f)

In [33]:
with open(OUTPUT_PATH, "r") as f:
    anns = json.load(f)